# Vision Brand Evaluation

Review brand logo detection dataset and metrics.

Steps:
- Inspect processed YOLO dataset.
- Load label configuration.
- Review vision metrics outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from pathlib import Path

summary = {
    'yolo': {},
    'metrics': {},
}

brand_root = REPO_ROOT / 'data' / 'processed' / 'brand_yolo'
if brand_root.exists():
    images_root = brand_root / 'images'
    labels_root = brand_root / 'labels'
    for split in ['train', 'val']:
        img_dir = images_root / split
        lbl_dir = labels_root / split
        img_files = [p for p in img_dir.rglob('*') if p.is_file()] if img_dir.exists() else []
        lbl_files = [p for p in lbl_dir.rglob('*') if p.is_file()] if lbl_dir.exists() else []
        summary['yolo'][split] = {
            'images': len(img_files),
            'labels': len(lbl_files),
        }
        print(split, summary['yolo'][split])
else:
    print('Missing:', brand_root)


In [ ]:
# Parse brands.yaml for class list if available.
yaml_path = REPO_ROOT / 'data' / 'processed' / 'brand_yolo' / 'brands.yaml'
if yaml_path.exists():
    try:
        import yaml
        data = yaml.safe_load(yaml_path.read_text())
    except Exception:
        data = None
    if data and 'names' in data:
        summary['yolo']['class_count'] = len(data['names'])
        summary['yolo']['classes_preview'] = data['names'][:10]
        print('Class count:', summary['yolo']['class_count'])
        print('Classes:', summary['yolo']['classes_preview'])
    else:
        print('Could not parse classes from brands.yaml')
else:
    print('Missing:', yaml_path)


In [ ]:
# Load vision metrics if present.
metrics_paths = [
    REPO_ROOT / 'experiments' / 'vision' / 'metrics.csv',
    REPO_ROOT / 'experiments' / 'vision' / 'metrics.json',
]
for path in metrics_paths:
    if not path.exists():
        continue
    if path.suffix == '.json':
        data = json.loads(path.read_text(encoding='utf-8'))
        summary['metrics'][path.name] = data
        print(path.name, data)
    else:
        import pandas as pd
        df = pd.read_csv(path)
        summary['metrics'][path.name] = {
            'rows': int(df.shape[0]),
            'cols': int(df.shape[1]),
        }
        print(path.name, df.head(5))


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_vision_brand_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
